In [ ]:
import pandas as pd

store = pd.read_csv('../data/SuperstoreSample.csv', encoding='cp1252',dtype={'Order ID': str, 'Customer ID': str, 'Postal Code': str})

# Displaying rows and columns 
print(f"Store Data CSV → Rows: {store.shape[0]}, Columns: {store.shape[1]}")
print(store.dtypes)


In [ ]:
#Drop unnecessary columns
store = store.drop(columns=['Row ID'])
store = store.drop(columns=['Customer Name'])

1.'Row_ID' column was excluded from analysis because it does not provide analytical values, it serves only as a unique identifier.


2.'Customer Name' column was excluded becuase it does not provide analytical value and duplicates the information represented by Customer ID. 'Customer ID' column was kept for potential grouping and customer-level analysis.

In [ ]:
# Converting Order Date to datetime format
store['Order Date'] = pd.to_datetime(store['Order Date'])

#Converting Ship Date to datetime format
store['Ship Date']  = pd.to_datetime(store['Ship Date'])

print(store.dtypes)

In [ ]:
#Checking for null values
store.isnull().sum()

In [ ]:
#Checking for duplicates
store.duplicated().sum()

In [ ]:
#Creating additional colums for time analysis 
store['Order Month'] = store['Order Date'].dt.month
store['Order Year'] = store['Order Date'].dt.year

#Calc shipping duration (days between order and delivery)
store['Shipping Duration'] = (store['Ship Date'] - store['Order Date']).dt.days

#Calc profit margin as %
store['Profit Margin'] =  (store['Profit'] / store['Sales']) * 100

#Categorising profit status as 'Profit' or 'Loss'
store['Profit Status'] = store['Profit'].apply(lambda x: 'Loss' if x < 0 else 'Profit')


print(store.columns)

Several transactions show negative values within the 'Profit' column, indicating loss-making sales. As expected, these cases will result in negative 'Profit Margin(%)' values. Further analysis will focus on identifying drivers such as discounts or inefficient pricing.

In [ ]:
#Validation
store.isnull().sum()

In [ ]:
#More Validations 
print(store['Profit Status'].value_counts())
print(f"Total Rows: {store.shape[0]}")
print(f"Rows with negative shipping duration: {store[store['Shipping Duration'] < 0].shape[0]}")

In [ ]:
store.head()

In [ ]:
#Additional column for discount percentage
store['Discount %'] = store['Discount'] * 100

In [ ]:
#Validation
print(store[['Discount', 'Discount %']].head())

In [ ]:
store[['Sales', 'Profit', 'Discount', 'Shipping Duration', 'Profit Margin', 'Discount %']].describe()

1.Sales values are right-skewed, with a mean (229.86) larger than the median (54.49). This shows that most transactions are of low value, while a small number of high-value sales bring the overall sales average upward. This suggests the presence of outliers.


2.Profit values show large variability. Average profit is 28.66, the median is at 8.67, which shows that most transactions generate relatively small profits. Max(8399.976000) and Min(-6599.978000) values suggest inconsistent profitability and influence of extreme outliers.


3.Shipping duration looks relatively stable.


4.Discount % values - most transactions either have no discount or discount rate of around 20%, while a smaller number of transactions receive significantly higher discounts (may contribute to fluctuations in profitability)


5.Profit margins range from -275% to 50%, with an average of 12%. The minimum value (-275.000000) indicates the presence of extreme loss-making transactions. This suggests that certain transactions are heavily impacted by factors such as discounts or high costs.

In [ ]:
#Calc unit price 
store['Unit Price'] = store['Sales'] / store['Quantity']
store['Unit Price'].describe()

'Unit Price' shows high variability, could possibly be a reflection of the differences across product sub-categories. This relationship will be explored further.

In [ ]:
store.to_csv('../data/superstore_cleaned.csv', index=False)